In [19]:
import os
from urllib import response

from PyInstaller.compat import system
from langgraph import graph
from langgraph.graph import StateGraph,END,START
from typing import Dict,TypedDict,List,Union,Annotated,Sequence
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage,ToolMessage,SystemMessage
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tool_node


In [20]:
load_dotenv()

document_content = []
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def update(content:str) -> str:
    "update document"
    global document_content
    document_content = content
    return f"Document has been updated: \n{document_content}"

@tool
def save(filename:str) -> str:
    "save document"
    global document_content

    if not filename.endswith(".txt"):
        filename += ".txt"

    try:
        with open(filename,"w") as file:
            file.write(document_content)
        print(f"Document has been saved to {filename}")
        return f"Document has been saved to {filename}"

    except Exception as e:
        return f"Error while saving document: {str(e)}"

In [21]:
tools = [update,save]

llm = ChatDeepSeek(model="deepseek-chat", temperature=0).bind_tools(tools)

def our_agent(state:AgentState) -> AgentState:
    system_prompt = SystemMessage(content= F"""You are Drafter, a helpful writing assistant.
    You are going to help the user update and modify content.

    -If the user wants to update or modify content, use the 'update' tool with the complete content.
    -If the user wants to save and finish, you need to use the 'save' tool.
    -Make sure to always show the current document state after modifications.

    The current document content is: {document_content}""")

    if not state['messages']:
        user_input = input()
        user_message = HumanMessage(content=user_input)

    else:
        user_input = input()
        print(f"\n USER:{user_input}")
        user_message = HumanMessage(content=user_input)


    all_messages = [system_prompt] + list(state['messages']) + [user_message]
    response = llm.invoke(all_messages)

    print(f"\n🤖 AI: {response.content}")

    if hasattr(response, "tool_calls") and response.tool_calls:
      print(f"🔧 USING TOOLS: {[tc['name'] for tc in response.tool_calls]}")

    return {'messages': list(state['messages'])+[user_message,response]}


In [22]:
def should_continue(state:AgentState) :
    messages = state['messages']

    if not messages:
        return 'continue'
    for messages in reversed(messages):
        if(isinstance(messages,ToolMessage) and
        "saved" in messages.content.lower() and
        "document" in messages.content.lower()):
            return "end"
    return "continue"

def print_messages(messages):
    """Function I made to print the messages in a more readable format"""
    if not messages:
        return

    for message in messages[-3:]:
        if isinstance(message, ToolMessage):
            print(f"\n🔧 TOOL RESULT: {message.content}")


In [23]:
graph = StateGraph(AgentState)

graph.add_node("agent",our_agent)
graph.add_node("tools",ToolNode(tools))

graph.set_entry_point("agent")
graph.add_edge("agent","tools")

graph.add_conditional_edges("tools",should_continue,{"continue":"agent","end":END})

app= graph.compile()

In [24]:
def run_document_agent():
    print("\n ===== DRAFTER =====")
    state = {"messages": []}

    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])

    print("\n ===== DRAFTER FINISHED =====")

if __name__ == "__main__":
    run_document_agent()



 ===== DRAFTER =====

🤖 AI: 我来帮您整理文档。不过目前文档内容是空的，请问您想整理什么内容呢？请提供需要整理的文本，我会帮您整理成结构清晰的文档。

 USER:写一个email


🤖 AI: 我来帮您写一封邮件。不过为了写出合适的邮件，我需要了解一些信息：

1. **收件人是谁？**（例如：同事、客户、上司、朋友等）
2. **邮件目的是什么？**（例如：请假、申请、感谢、道歉、通知、跟进等）
3. **需要包含哪些关键信息？**（例如：时间、地点、具体事项等）
4. **语气风格？**（正式 / 半正式 / 随意）

如果您没有特别要求，我可以先帮您写一封**通用的正式商务邮件模板**作为起点。请告诉我您的具体需求，或者直接说"先写个通用模板"，我就开始。

 USER:随便编一个


🤖 AI: 好的，我帮您编一封通用的商务邮件。
🔧 USING TOOLS: ['update']

🔧 TOOL RESULT: Document has been updated: 
主题：关于下周一项目启动会议的确认

尊敬的王经理：

您好！

感谢您一直以来对本项目的支持与关注。为推进项目顺利开展，我们希望就下周一（3月18日）上午10:00召开的项目启动会议与您进行确认。

会议安排如下：

- 时间：3月18日（周一）上午10:00 - 11:00
- 地点：公司三楼会议室A
- 议题：
  1. 项目目标与范围说明
  2. 团队成员分工介绍
  3. 关键时间节点讨论
  4. 下一步行动计划

如您时间不便，烦请告知，我们可以另行调整。期待您的回复。

顺祝商祺！

李明
市场部
电话：138-0000-0000
邮箱：liming@example.com
2024年3月12日


 USER:保存结果吧


🤖 AI: 好的，我来保存文档。
🔧 USING TOOLS: ['save']

🔧 TOOL RESULT: Document has been updated: 
主题：关于下周一项目启动会议的确认

尊敬的王经理：

您好！

感谢您一直以来对本项目的支持与关注。为推进项目顺利开展，我们希望就下周一（3月18日）上午10:00召开的项目启动会议与您进行确认。

会议安排如下：

- 时间：3月18日（周一）上午1